# Phase 4 – Court Judgment Prediction (CJPE): Master Notebook *(v3 – LP-FT rebuild)*

## Why v2 scored 0.334
The v2 Run 1 never left its starting point: training loss stayed at ln 2 ≈ 0.692, Dev AUC was below 0.5, and the
fixed threshold grid ended up predicting class 1 for every test case. Three things in v2 combined to make that likely:

1. **The head started from scratch on top of adapters that also started from noise** (random `lora_B`, BiLSTM, two pooling
   paths, a skip layer), with batch size 1 and under one epoch of data. A randomly initialised head on a slowly
   moving backbone is the classic recipe for collapsing to the class prior.
2. **No check that the backbone features carry any signal**, so there was no way to tell whether the head or the features were at fault.
3. **The threshold search used a fixed 0.05–0.95 grid**: when probabilities are compressed around 0.5, every threshold
   gives the same single class.

## What v3 does instead (LP-FT: linear-probe, then fine-tune)
| Step | What happens | Why |
|---|---|---|
| Sanity | Next-token **perplexity of each Phase-1 LM on Dev text** + fp16 overflow test | If perplexity is in the thousands, the architecture code/tokenizer does not reproduce Phase 1 and nothing downstream can work. You see it in 10 seconds instead of after 3 hours. |
| Features | Each doc = 1 head chunk + 3 tail chunks of 512 tokens, **identical at train and eval**. Per chunk: mean of the final layer **and** of block 8 | Middle layers of small LMs usually carry more task signal than the next-token layer. |
| Check 1 | Logistic regression on the frozen features | Answers the "cause 1 vs cause 2" question automatically on every run. |
| Stage A | Features for all 37k training docs are extracted **once**, then the run's head trains for up to 40 epochs on them (minutes) | The head sees the whole training set many times instead of 0.67 epochs. |
| Stage B | Adapters are injected as the **identity** (`lora_B = 0`, AdaLoRA `Λ = 0`), so fine-tuning starts exactly at the Stage-A solution; the best Dev checkpoint is kept, so Stage B can only help | Standard LP-FT (Kumar et al., 2022). |
| Threshold | Candidates are **quantiles** of the scores (5 %–95 %) | The chosen threshold always predicts both classes. |
| Persistence | Each run saves Dev/Test probabilities to `preds/` | Runs can be split across Kaggle sessions; Run 5 picks them up. |

## How to run on Kaggle (GPU T4 or P100)
1. Set `DRY_RUN = True` in cell 1 and run everything once (~30 min). This catches any error cheaply.
2. Set `DRY_RUN = False`. One full run takes roughly 2 h (≈35 min feature extraction, ≤75 min fine-tuning, the rest
   evaluation). Runs 1–4 together are about 9 h, so either run everything in one 12 h session or split it:
   `RUNS_TO_EXECUTE = [1, 2]` in session 1 → save version → add its output as a dataset → `RUNS_TO_EXECUTE = [3, 4, 5]`
   with `PREV_OUT_DIRS = ['/kaggle/input/<that-output>/preds']`.
3. Read the first lines each run prints: **perplexity**, **Check 1 AUC**, **Stage A Dev F1**. They tell you where the
   ceiling is before any long training starts.

Each run writes a CodaBench ZIP with `prediction.json` at its root.


In [ ]:
# 1. Environment, CONFIG, hardware
import os, gc, sys, json, math, time, random, glob, zipfile, pickle, warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.checkpoint import checkpoint
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

DRY_RUN = False

# ------------------------------- CONFIG ---------------------------------------
RUNS_TO_EXECUTE = [1, 2, 3, 4, 5]
PREV_OUT_DIRS = []          # e.g. ['/kaggle/input/cjpe-run1-output/preds'] to reuse earlier sessions' results

# Backbone geometry (must match the Phase-1 checkpoints)
D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 768, 12, 12, 4
FF_SWIGLU, FF_GELU, MAX_POS = 2560, 3072, 2048

# Document view. The SAME chunks are used for training and evaluation (no train/eval shift).
CHUNK_LEN   = 512
HEAD_CHUNKS = 1             # opening of the judgment: facts, issues
TAIL_CHUNKS = 3             # last 1,536 tokens: the court's reasoning
ENC_BS      = 16            # chunks per backbone forward pass during feature extraction
MID_BLOCK   = 7             # output of block 8/12 is mean-pooled alongside the final layer
GRAD_CKPT   = True

# Stage A (linear probe / head training on cached frozen features)
LP_EPOCHS, LP_BS, LP_LR, LP_WD, LP_PATIENCE = 40, 64, 1e-3, 0.05, 6

# Stage B (PEFT fine-tuning; starts from the Stage-A head, adapters start as the identity)
FT_MINUTES    = 75          # wall-clock budget per run for Stage B (0 = skip Stage B)
FT_ACCUM      = 16          # documents per optimiser step
FT_LR_PEFT    = 2e-4
FT_LR_HEAD    = 1e-4
FT_EVAL_EVERY = 3000        # documents between full-Dev evaluations
FT_PATIENCE   = 3
LR_LAYER_TOP, XI = 5e-5, 0.85   # Run 3: LR of the top block, geometric decay per block below it
KD_DOCS       = 15000       # training documents whose teacher outputs are cached for Run 4

LABEL_SMOOTH  = 0.05
CACHE_FEATS_TO_DISK = True  # reuse extracted features if a cell is re-run in the same session
USE_TFIDF_MODEL = False     # classical TF-IDF model as an extra ensemble member (only if the rules allow it)
SEED = 42
DRY_RUN_TRAIN_DOCS = 2000
if DRY_RUN:
    LP_EPOCHS, FT_MINUTES, FT_EVAL_EVERY, KD_DOCS = 10, 4, 400, 500
    CACHE_FEATS_TO_DISK = False

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
AMP_ON = True               # switched off per run if the backbone overflows in fp16 (checked automatically)
print(f"Device: {device} | AMP: {USE_AMP} ({AMP_DTYPE})")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

def amp_ctx():
    on = USE_AMP and AMP_ON
    return torch.autocast(device_type=device.type, dtype=AMP_DTYPE if on else torch.bfloat16, enabled=on)

def make_scaler():
    on = USE_AMP and AMP_ON and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler('cuda', enabled=on)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=on)

def free_mem():
    for name in ['_', '__', '___']:
        if name in globals():
            globals()[name] = None
    for a in ('last_traceback', 'last_value', 'last_type'):
        if hasattr(sys, a): setattr(sys, a, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./working"
OUT_DIR = os.path.join(WORK_DIR, "preds")
FEAT_DIR = os.path.join(WORK_DIR, "feats")
for d in (WORK_DIR, OUT_DIR, FEAT_DIR): os.makedirs(d, exist_ok=True)

# Hugging Face authentication (only needed if files are pulled from the Hub)
HF_TOKEN = None
if True:
    for _name in ("HF", "HF_TOKEN"):
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret(_name)
            if HF_TOKEN: break
        except Exception:
            pass
    HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        try:
            from huggingface_hub import login
            login(token=HF_TOKEN); print("✓ Hugging Face authentication successful.")
        except Exception as e:
            print(f"HF login note: {e}")

In [ ]:
# 2. Universal Checkpoint Resolver & Directory-to-.PT Converter

def find_checkpoint(candidates: List[str], glob_pattern: str = "") -> Optional[str]:
    for c in candidates:
        if os.path.exists(c):
            return c
    if glob_pattern:
        matches = sorted(glob.glob(glob_pattern, recursive=True))
        if matches:
            return matches[0]
    return None

def pack_directory_to_pt(source_dir: str, target_pt: str) -> str:
    print(f"-> Directory detected: '{source_dir}'. Repacking to valid .pt archive: '{target_pt}'...")
    files = [(os.path.join(r, f), os.path.relpath(os.path.join(r, f), source_dir))
             for r, _, fs in os.walk(source_dir) for f in fs]
    has_top_dir = any(rel.endswith('data.pkl') and os.sep in rel for _, rel in files)
    prefix = '' if has_top_dir else 'archive/'
    with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
        for full, rel in files:
            zf.write(full, arcname=prefix + rel)
    print(f"✓ Created checkpoint archive: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")
    return target_pt

ARCH4_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/phase1-arch4/decoder_lm.pt',
    './decoder_lm.pt',
    'd:/D_Downloads/Phase-1/decoder_lm.pt',
    'd:/D_Downloads/decoder_lm.pt'
]
ARCH3_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm',
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/phase1-arch3/decoder_lm.pt',
    './decoder_lm_arch3.pt'
]
ARCH2_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch2/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch2/decoder_lm.pt',
    '/kaggle/input/phase1-arch2/decoder_lm.pt',
    './decoder_lm_arch2.pt',
    'd:/D_Downloads/Phase-1/decoder_lm.pt'
]
ARCH1_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/phase1-arch1/decoder_lm.pt',
    './decoder_lm_arch1.pt'
]
TOKENIZER_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json',
    '/kaggle/input/custom_bpe.json',
    './custom_bpe.json',
    'd:/D_Downloads/Phase-1/custom_bpe.json'
]

ARCH4_RAW = find_checkpoint(ARCH4_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch4/**/decoder_lm*") or find_checkpoint([], "/kaggle/input/**/decoder_lm.pt")
ARCH3_RAW = find_checkpoint(ARCH3_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch3/**")
ARCH2_RAW = find_checkpoint(ARCH2_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch2/**")
ARCH1_RAW = find_checkpoint(ARCH1_CANDIDATES, "/kaggle/input/**/phase1-arch1/**")
TOKENIZER_PATH = find_checkpoint(TOKENIZER_CANDIDATES, "/kaggle/input/**/custom_bpe.json")

print("Discovered Raw Checkpoint Paths:")
print(f"  Arch 4 Raw : {ARCH4_RAW}")
print(f"  Arch 3 Raw : {ARCH3_RAW}")
print(f"  Arch 2 Raw : {ARCH2_RAW}")
print(f"  Arch 1 Raw : {ARCH1_RAW}")
print(f"  Tokenizer  : {TOKENIZER_PATH}")

ARCH4_CHECKPOINT = ARCH4_RAW
if ARCH4_RAW and os.path.isdir(ARCH4_RAW):
    ARCH4_CHECKPOINT = pack_directory_to_pt(ARCH4_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch4.pt'))

ARCH3_CHECKPOINT = ARCH3_RAW
if ARCH3_RAW and os.path.isdir(ARCH3_RAW):
    ARCH3_CHECKPOINT = pack_directory_to_pt(ARCH3_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch3.pt'))

ARCH2_CHECKPOINT = ARCH2_RAW
if ARCH2_RAW and os.path.isdir(ARCH2_RAW):
    ARCH2_CHECKPOINT = pack_directory_to_pt(ARCH2_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch2.pt'))

ARCH1_CHECKPOINT = ARCH1_RAW
if ARCH1_RAW and os.path.isdir(ARCH1_RAW):
    ARCH1_CHECKPOINT = pack_directory_to_pt(ARCH1_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch1.pt'))

In [ ]:
# 3. Fast Byte-Level BPE Tokenizer with Word-Level Caching
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class FastByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}
        self._word_cache = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def _word_ids(self, w):
        r = self._word_cache.get(w)
        if r is None:
            unk = self.stoi.get('<unk>', 1)
            r = [self.stoi.get(s, unk) for s in self._apply_merges_to_word(w)]
            self._word_cache[w] = r
        return r

    def encode(self, text, max_len=None):
        ids = [i for w in text.split() for i in self._word_ids(w)]
        return ids[:max_len] if max_len else ids

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']
        t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

if not TOKENIZER_PATH or not os.path.exists(TOKENIZER_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOKENIZER_PATH = hf_hub_download(repo_id="Exploration-Lab/CS-781-Capstone", filename="Phase_1_LM/custom_bpe.json", repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        print(f"Tokenizer download note: {e}")

tokenizer = FastByteBPETokenizer.load(TOKENIZER_PATH) if (TOKENIZER_PATH and os.path.exists(TOKENIZER_PATH)) else FastByteBPETokenizer()
PAD_ID = tokenizer.stoi.get('<pad>', 0)
SEP_ID = tokenizer.stoi.get('</s>', 3)
print(f"✓ Tokenizer ready: {tokenizer.vocab_size} vocab tokens | PAD_ID={PAD_ID}, SEP_ID={SEP_ID}")

In [ ]:
# 4. Phase 4 CJPE Dataset Loading & Class Balancing (unchanged except the DRY_RUN subset)
def get_dataset_file(filename: str) -> str:
    local_hits = glob.glob(f"/kaggle/input/**/{Path(filename).name}", recursive=True)
    if local_hits:
        return local_hits[0]
    local_hits2 = glob.glob(f"**/{Path(filename).name}", recursive=True)
    if local_hits2:
        return local_hits2[0]
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id="Exploration-Lab/CS-781-Capstone", filename=filename, repo_type="dataset", token=HF_TOKEN)

print("Resolving Phase 4 CJPE dataset files...")
TRAIN_FILE = get_dataset_file("Phase_4_CJPE/Train/train.json")
DEV_FILE   = get_dataset_file("Phase_4_CJPE/Train/dev.json")
TEST_FILE  = get_dataset_file("Phase_4_CJPE/Test/test.json")
print(f"  Train : {TRAIN_FILE}")
print(f"  Dev   : {DEV_FILE}")
print(f"  Test  : {TEST_FILE}")

def load_cjpe_data(file_path: str, is_test: bool = False) -> List[Dict[str, Any]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if isinstance(data, dict):
        if 'data' in data and isinstance(data['data'], list):
            raw_list = data['data']
        else:
            raw_list = [{'id': k, **v} if isinstance(v, dict) else {'id': k, 'text': v} for k, v in data.items()]
    elif isinstance(data, list):
        raw_list = data
    else:
        raise ValueError(f"Unsupported format in {file_path}")

    cleaned = []
    for idx, item in enumerate(raw_list):
        cid = str(item.get('id', item.get('case_id', f'case_{idx:04d}')))
        text = str(item.get('text', item.get('case_text', ''))).strip()
        label = item.get('label', item.get('judgment', item.get('outcome', None)))
        if not is_test and label is not None:
            label = int(label)
        cleaned.append({'id': cid, 'text': text, 'label': label})
    return cleaned

train_records = load_cjpe_data(TRAIN_FILE, is_test=False)
dev_records   = load_cjpe_data(DEV_FILE, is_test=False)
test_records  = load_cjpe_data(TEST_FILE, is_test=True)

if DRY_RUN:
    train_records = random.Random(SEED).sample(train_records, min(DRY_RUN_TRAIN_DOCS, len(train_records)))
    print(f"DRY_RUN: training on a random subset of {len(train_records)} documents.")

print(f"Loaded records -> Train: {len(train_records)} | Dev: {len(dev_records)} | Test: {len(test_records)}")

train_labels = [r['label'] for r in train_records if r['label'] is not None]
counts = Counter(train_labels)
print(f"Train Class Distribution: Class 0 = {counts.get(0, 0)} ({counts.get(0, 0)/len(train_labels)*100:.1f}%), Class 1 = {counts.get(1, 0)} ({counts.get(1, 0)/len(train_labels)*100:.1f}%)")

total_samples = len(train_labels)
w0 = total_samples / (2.0 * max(1, counts.get(0, 1)))
w1 = total_samples / (2.0 * max(1, counts.get(1, 1)))
class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)
print(f"Balanced Class Weights: Class 0 = {w0:.3f}, Class 1 = {w1:.3f}")

In [ ]:
# 5. Head + tail chunking and a compact token store
#    Every document becomes the SAME set of chunks at train and eval time:
#    HEAD_CHUNKS x 512 tokens from the start + TAIL_CHUNKS x 512 tokens ending at the last token.
#    Long documents are only tokenised at their two ends (each word gives >= 1 token), which
#    makes pre-tokenisation several times faster than encoding the full judgment.
K_DOC = HEAD_CHUNKS + TAIL_CHUNKS
UNK_ID = tokenizer.stoi.get('<unk>', 1)

def _enc_words(words):
    return [i for w in words for i in tokenizer._word_ids(w)]

def split_tokens(toks: List[int]) -> List[List[int]]:
    L, H, T = CHUNK_LEN, HEAD_CHUNKS, TAIL_CHUNKS
    n = len(toks)
    if n == 0:
        return [[SEP_ID]]
    if n <= L:
        return [toks]
    if n <= K_DOC * L:
        starts = list(range(0, n - L + 1, L))
        if starts[-1] != n - L:
            starts.append(n - L)                      # last chunk ends exactly at the document end
        if len(starts) > K_DOC:
            starts = starts[:H] + starts[len(starts) - T:]
        return [toks[s:s + L] for s in starts]
    starts = [i * L for i in range(H)] + [n - (T - j) * L for j in range(T)]
    return [toks[s:s + L] for s in starts]

def doc_to_chunks(text: str) -> List[List[int]]:
    L, H, T = CHUNK_LEN, HEAD_CHUNKS, TAIL_CHUNKS
    words = text.split()
    if len(words) > K_DOC * L:                       # tokens >= words, so head and tail cannot overlap
        head = _enc_words(words[:H * L])[:H * L] if H > 0 else []
        tail = _enc_words(words[-T * L:])[-T * L:]
        return [head[i * L:(i + 1) * L] for i in range(H)] + [tail[j * L:(j + 1) * L] for j in range(T)]
    return split_tokens(_enc_words(words))

class TokStore:
    """All chunks of a split in one int16 matrix; doc i owns rows ptr[i]:ptr[i+1]."""
    def __init__(self, records=None, name=''):
        if records is None:
            return
        t0 = time.time()
        rows, lens, ptr, n_unk, n_tok, n_words = [], [], [0], 0, 0, []
        for r in records:
            ch = doc_to_chunks(r['text'])
            for c in ch:
                rows.append(c); lens.append(len(c))
                n_unk += sum(1 for x in c if x == UNK_ID); n_tok += len(c)
            ptr.append(ptr[-1] + len(ch)); n_words.append(len(r['text'].split()))
        dt = np.int16 if tokenizer.vocab_size < 32000 else np.int32
        self.flat = np.full((len(rows), CHUNK_LEN), PAD_ID, dtype=dt)
        for j, c in enumerate(rows):
            self.flat[j, :len(c)] = c
        self.lens = np.array(lens, dtype=np.int64)
        self.ptr = np.array(ptr, dtype=np.int64)
        self.ids = [r['id'] for r in records]
        self.labels = np.array([(-1 if r['label'] is None else int(r['label'])) for r in records], dtype=np.int64)
        nw = np.array(n_words)
        print(f"✓ [{name}] {len(records)} docs -> {len(rows)} chunks in {time.time()-t0:.0f}s | "
              f"words/doc median={int(np.median(nw))}, p90={int(np.percentile(nw, 90))} | "
              f"chunks/doc={len(rows)/max(1,len(records)):.2f} | <unk> rate={n_unk/max(1,n_tok)*100:.2f}%")
        if n_unk / max(1, n_tok) > 0.05:
            print("  ⚠⚠ more than 5% <unk> tokens: the tokenizer file does not match this text -> features will be poor.")

    def __len__(self):
        return len(self.ids)

    def get(self, i: int):
        a, b = self.ptr[i], self.ptr[i + 1]
        ids = torch.from_numpy(self.flat[a:b].astype(np.int64))
        mask = torch.arange(CHUNK_LEN)[None, :] < torch.from_numpy(self.lens[a:b])[:, None]
        return ids, mask

    def subset(self, idx) -> 'TokStore':
        s = TokStore()
        rows = np.concatenate([np.arange(self.ptr[i], self.ptr[i + 1]) for i in idx])
        s.flat, s.lens = self.flat[rows], self.lens[rows]
        s.ptr = np.concatenate([[0], np.cumsum([self.ptr[i + 1] - self.ptr[i] for i in idx])]).astype(np.int64)
        s.ids = [self.ids[i] for i in idx]; s.labels = self.labels[np.asarray(idx)]
        return s

def build_store(records, name):
    path = os.path.join(WORK_DIR, f"tok_{name}_{len(records)}_{CHUNK_LEN}_{HEAD_CHUNKS}_{TAIL_CHUNKS}.pkl")
    if os.path.exists(path):
        with open(path, 'rb') as f:
            s = pickle.load(f)
        print(f"✓ [{name}] loaded token cache ({len(s)} docs)")
        return s
    s = TokStore(records, name)
    with open(path, 'wb') as f:
        pickle.dump(s, f)
    return s

train_store = build_store(train_records, 'train')
dev_store   = build_store(dev_records, 'dev')
test_store  = build_store(test_records, 'test')
dev_labels  = dev_store.labels
test_ids    = test_store.ids
print(f"Dev label balance: {np.bincount(dev_labels, minlength=2)}")

In [ ]:
# 6. Shared engine
#    - checkpoint loading (key remapping, % restored) + backbone sanity check (perplexity, fp16 safety)
#    - FeatureEncoder: per chunk, mean-pool the final layer AND an intermediate block
#    - document heads (BiLSTM / gated-attention / temporal-conv) that work on chunk vectors
#    - Stage A: train the head on cached frozen features (fast, many epochs)
#    - Stage B: PEFT fine-tuning starting from the Stage-A head (adapters start as the identity)
#    - threshold calibration, packaging, result persistence
RUN_PREDICTIONS_DEV, RUN_PREDICTIONS_TEST, RUN_METRICS_SUMMARY = {}, {}, {}

# ---------------------------- checkpoint loading ------------------------------
_ALIASES = [('.n1.', '.norm1.'), ('.n2.', '.norm2.'), ('ln1', 'norm1'), ('ln2', 'norm2'),
            ('w_gate', 'gate_proj'), ('w_up', 'up_proj'), ('w_down', 'down_proj'),
            ('wq.', 'q_proj.'), ('wk.', 'k_proj.'), ('wv.', 'v_proj.'), ('wo.', 'out_proj.'), ('o_proj', 'out_proj'),
            ('final_norm', 'norm_f'), ('ln_f', 'norm_f'), ('embed_tokens', 'tok_emb'), ('wte', 'tok_emb'),
            ('layers.', 'blocks.'), ('self_attn', 'attn'), ('mlp.', 'ffn.')]
_PREFIXES = ('module.', '_orig_mod.', 'model.', 'backbone.', 'transformer.')

def _strip(k: str) -> str:
    changed = True
    while changed:
        changed = False
        for pre in _PREFIXES:
            if k.startswith(pre):
                k = k[len(pre):]; changed = True
    return k

def _key_variants(k: str):
    base = _strip(k); fwd, rev = base, base
    for a, b in _ALIASES:
        fwd = fwd.replace(a, b); rev = rev.replace(b, a)
    return [k, base, fwd, rev]

def load_pretrained(backbone: nn.Module, path: Optional[str], name: str) -> float:
    if not path or not os.path.exists(path):
        print(f"⚠⚠ [{name}] checkpoint NOT FOUND -> backbone is RANDOMLY initialised (scores will be poor).")
        return 0.0
    raw = torch.load(path, map_location='cpu', weights_only=False)
    sd = raw
    if isinstance(raw, dict):
        for key in ('model_state_dict', 'model', 'state_dict'):
            if key in raw and isinstance(raw[key], dict):
                sd = raw[key]; break
    model_sd = backbone.state_dict()
    mapped = {}
    for k, v in sd.items():
        if not torch.is_tensor(v): continue
        for cand in _key_variants(k):
            if cand in model_sd and tuple(model_sd[cand].shape) == tuple(v.shape):
                mapped[cand] = v; break
    # output projection of the LM (used only for the perplexity sanity check)
    emb_shape = tuple(backbone.tok_emb.weight.shape)
    for k, v in sd.items():
        if torch.is_tensor(v) and _strip(k).endswith(('lm_head.weight', 'head.weight', 'output.weight')) and tuple(v.shape) == emb_shape:
            backbone._lm_w = v.detach().clone(); break
    if 'tok_emb.weight' not in mapped and getattr(backbone, '_lm_w', None) is not None:
        mapped['tok_emb.weight'] = backbone._lm_w
    backbone.load_state_dict(mapped, strict=False)
    tot = sum(p.numel() for p in backbone.parameters())
    got = sum(p.numel() for n, p in backbone.named_parameters() if n in mapped)
    frac = got / max(1, tot)
    print(("✓ " if frac > 0.9 else "⚠⚠ ") + f"[{name}] loaded {len(mapped)} tensors | {frac*100:.1f}% of backbone parameters restored")
    if frac <= 0.9:
        missing = [n for n, _ in backbone.named_parameters() if n not in mapped]
        print(f"   WARNING: key mismatch! Missing e.g.: {missing[:6]}\n   Checkpoint keys e.g.: {list(sd.keys())[:6]}")
    return frac

def disable_dropout(module: nn.Module):
    # Backbone dropout off: features are deterministic, and the frozen-feature head (Stage A)
    # sees exactly what the fine-tuned model produces at the start of Stage B.
    for m in module.modules():
        if isinstance(m, nn.Dropout): m.p = 0.0
        if hasattr(m, 'dropout') and isinstance(getattr(m, 'dropout'), float): m.dropout = 0.0

@torch.no_grad()
def backbone_sanity(backbone: nn.Module, name: str, n_chunks: int = 8):
    """Next-token perplexity of the Phase-1 LM on Dev chunks. A value in the tens means the
    architecture code, the checkpoint and the tokenizer all agree with Phase 1. Thousands means the
    hidden states are garbage, and no classifier on top of them can learn (the most likely cause of
    an AUC ~0.5). Also verifies that fp16 autocast does not overflow; if it does, AMP is turned off."""
    global AMP_ON
    backbone.eval()
    W = getattr(backbone, '_lm_w', None)
    if W is None:
        W = backbone.lm_head.weight if hasattr(backbone, 'lm_head') else backbone.tok_emb.weight
    W = W.to(device).float()
    full = np.where(dev_store.lens == CHUNK_LEN)[0][:n_chunks]
    rows = full if len(full) else np.arange(min(n_chunks, len(dev_store.lens)))
    ids = torch.from_numpy(dev_store.flat[rows].astype(np.int64)).to(device)
    mask = (torch.arange(CHUNK_LEN, device=device)[None] < torch.from_numpy(dev_store.lens[rows]).to(device)[:, None])
    def ppl(use_amp):
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            h = backbone(ids)
        if not torch.isfinite(h).all():
            return float('nan')
        logits = h.float() @ W.t()
        ce = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1), reduction='none')
        m = mask[:, 1:].reshape(-1).float()
        return float(torch.exp((ce * m).sum() / m.sum()))
    p32 = ppl(False)
    msg = f"[{name}] sanity: Dev next-token perplexity (fp32) = {p32:.1f}"
    if USE_AMP:
        pam = ppl(True)
        msg += f" | with {str(AMP_DTYPE).split('.')[-1]} autocast = {pam:.1f}"
        if not math.isfinite(pam) or abs(math.log(max(pam, 1e-6)) - math.log(max(p32, 1e-6))) > 0.15:
            AMP_ON = False
            msg += "  -> mixed precision is unsafe for this backbone, running it in fp32"
    print(msg)
    if not math.isfinite(p32) or p32 > 300:
        print(f"⚠⚠ [{name}] perplexity is far above the Phase-1 value. The architecture code / tokenizer here does not\n"
              f"   reproduce the Phase-1 model, so its features carry little signal. Fix this before training.")
    del W
    return p32

# ---------------------------- chunk feature encoder ---------------------------
def pool_pair(last: torch.Tensor, mid: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    mf = m.unsqueeze(-1).float()
    n = mf.sum(1).clamp(min=1.0)
    last, mid = last.float(), mid.float()
    mid = mid * torch.rsqrt(mid.pow(2).mean(-1, keepdim=True) + 1e-6)     # un-normalised residual stream -> RMS-normalise
    return torch.cat([(last * mf).sum(1) / n, (mid * mf).sum(1) / n], dim=-1)

class FeatureEncoder(nn.Module):
    """backbone -> (K, 2*D) chunk vectors: masked mean of the final layer and of block MID_BLOCK."""
    def __init__(self, backbone: nn.Module, mid_block: int = MID_BLOCK):
        super().__init__()
        self.backbone = backbone
        self.out_dim = 2 * backbone.tok_emb.embedding_dim
        self._mid = None
        backbone.blocks[mid_block].register_forward_hook(self._grab)
    def _grab(self, module, inp, out):
        self._mid = out[0] if isinstance(out, tuple) else out
    def _run(self, ids):
        last = self.backbone(ids); mid = self._mid; self._mid = None
        return last, mid
    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        outs = []
        for s in range(0, ids.size(0), ENC_BS):
            x, m = ids[s:s + ENC_BS], mask[s:s + ENC_BS]
            if self.training and GRAD_CKPT and torch.is_grad_enabled():
                last, mid = checkpoint(self._run, x, use_reentrant=False)
            else:
                last, mid = self._run(x)
            outs.append(pool_pair(last, mid, m))
        return torch.cat(outs, 0)

def to_padded(flat_feats: torch.Tensor, store: TokStore):
    N, Fd = len(store), flat_feats.size(-1)
    X = torch.zeros(N, K_DOC, Fd, dtype=flat_feats.dtype); M = torch.zeros(N, K_DOC, dtype=torch.bool)
    for i in range(N):
        a, b = int(store.ptr[i]), int(store.ptr[i + 1])
        X[i, :b - a] = flat_feats[a:b]; M[i, :b - a] = True
    return X, M

@torch.no_grad()
def extract_features(encoder: FeatureEncoder, store: TokStore, desc: str = '', cache_key: Optional[str] = None):
    path = os.path.join(FEAT_DIR, f"{cache_key}.pt") if (cache_key and CACHE_FEATS_TO_DISK) else None
    if path and os.path.exists(path):
        d = torch.load(path); print(f"  loaded cached features {cache_key}"); return d['X'], d['M']
    was_training = encoder.training
    encoder.eval()
    Nc = store.flat.shape[0]
    out = torch.empty(Nc, encoder.out_dim, dtype=torch.float16)
    t0, ar = time.time(), torch.arange(CHUNK_LEN, device=device)[None]
    step, every = ENC_BS, max(1, Nc // 5); next_report = every
    for s in range(0, Nc, step):
        ids = torch.from_numpy(store.flat[s:s + step].astype(np.int64)).to(device, non_blocking=True)
        mask = ar < torch.from_numpy(store.lens[s:s + step]).to(device)[:, None]
        with amp_ctx():
            f = encoder(ids, mask)
        out[s:s + len(ids)] = f.float().clamp(-6e4, 6e4).cpu().half()
        done = s + len(ids)
        if desc and Nc >= 2000 and done >= next_report and done < Nc:
            el = time.time() - t0; next_report += every
            print(f"    [{desc}] {done}/{Nc} chunks | {el/60:.1f} min elapsed, ~{el/done*(Nc-done)/60:.1f} min left")
    if desc:
        print(f"  ✓ [{desc}] features for {Nc} chunks in {(time.time()-t0)/60:.1f} min")
    if not torch.isfinite(out.float()).all():
        print(f"  ⚠⚠ [{desc}] non-finite features found")
    encoder.train(was_training)
    X, M = to_padded(out, store)
    if path: torch.save({'X': X, 'M': M}, path)
    return X, M

# ---------------------------- document heads ---------------------------------
def _inp_block(in_dim, d, drop):
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(drop), nn.Linear(in_dim, d), nn.GELU())

def _masked_softmax_pool(h, m, scorer):
    s = scorer(h).squeeze(-1).masked_fill(~m, -1e4)
    return (h * torch.softmax(s, dim=-1).unsqueeze(-1)).sum(1)

def _masked_mean(h, m):
    mf = m.unsqueeze(-1).float()
    return (h * mf).sum(1) / mf.sum(1).clamp(min=1.0)

def _masked_max(h, m):
    return h.masked_fill(~m.unsqueeze(-1), -1e4).max(1).values

class BiLSTMDocHead(nn.Module):
    """Run 1 / Run 4: chunk vectors -> BiLSTM over chunk order -> attention + max pooling -> linear."""
    def __init__(self, in_dim, d=384, hidden=256, drop=0.2):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.lstm = nn.LSTM(d, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(2 * hidden, 64), nn.Tanh(), nn.Linear(64, 1, bias=False))
        self.norm = nn.LayerNorm(4 * hidden); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(4 * hidden, 2)
    def forward(self, x, m):
        h = self.inp(x.float())
        lens = m.sum(1).clamp(min=1).cpu()
        packed = pack_padded_sequence(h, lens, batch_first=True, enforce_sorted=False)
        o, _ = self.lstm(packed)
        o, _ = pad_packed_sequence(o, batch_first=True, total_length=x.size(1))
        feat = self.norm(torch.cat([_masked_softmax_pool(o, m, self.attn), _masked_max(o, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class GatedAttnDocHead(nn.Module):
    """Run 2: gated (sigmoid x tanh) chunk units + chunk-position embeddings + self-attention across chunks."""
    def __init__(self, in_dim, d=384, drop=0.2, n_heads=4):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.g, self.f = nn.Linear(d, d), nn.Linear(d, d)
        self.pos = nn.Embedding(64, d)
        self.mha = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ln = nn.LayerNorm(d); self.score = nn.Linear(d, 1)
        self.norm = nn.LayerNorm(2 * d); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * d, 2)
    def forward(self, x, m):
        h = self.inp(x.float())
        h = torch.sigmoid(self.g(h)) * torch.tanh(self.f(h))
        h = h + self.pos(torch.arange(h.size(1), device=h.device).clamp(max=63))[None]
        a, _ = self.mha(h, h, h, key_padding_mask=~m, need_weights=False)
        h = self.ln(h + a)
        feat = self.norm(torch.cat([_masked_softmax_pool(h, m, self.score), _masked_mean(h, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class TemporalConvDocHead(nn.Module):
    """Run 3: 1-D convolution over the chunk sequence -> masked max + mean pooling."""
    def __init__(self, in_dim, d=384, hidden=256, drop=0.2):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.conv = nn.Conv1d(d, hidden, kernel_size=3, padding=1)
        self.norm = nn.LayerNorm(2 * hidden); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 2)
    def forward(self, x, m):
        h = self.inp(x.float()) * m.unsqueeze(-1).float()
        c = F.gelu(self.conv(h.transpose(1, 2))).transpose(1, 2)
        feat = self.norm(torch.cat([_masked_max(c, m), _masked_mean(c, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class DocModel(nn.Module):
    """Backbone (with adapters) + document head, for one document at a time in Stage B."""
    def __init__(self, encoder: FeatureEncoder, head: nn.Module):
        super().__init__()
        self.encoder, self.head = encoder, head
    def forward(self, ids, mask, return_feat=False):
        f = self.encoder(ids, mask)
        with torch.autocast(device_type=device.type, enabled=False):
            logits, feat = self.head(f.float().unsqueeze(0), torch.ones(1, f.size(0), dtype=torch.bool, device=f.device))
        return (logits, feat) if return_feat else logits

@torch.no_grad()
def head_predict(head: nn.Module, X, M, bs=256):
    was = head.training; head.eval()
    L, Fs = [], []
    for s in range(0, X.size(0), bs):
        lg, ft = head(X[s:s + bs].to(device).float(), M[s:s + bs].to(device))
        L.append(lg.float().cpu()); Fs.append(ft.float().cpu())
    head.train(was)
    return torch.cat(L), torch.cat(Fs)

def predict_store(model: DocModel, store: TokStore, desc=''):
    X, M = extract_features(model.encoder, store, desc)
    logits, feats = head_predict(model.head, X, M)
    return logits, feats

def softmax_np(logits: torch.Tensor) -> np.ndarray:
    return F.softmax(logits.float(), dim=-1).numpy()

# ---------------------------- threshold calibration ---------------------------
def calibrate_threshold(probs: np.ndarray, labels: np.ndarray) -> Tuple[float, float, float]:
    """Candidate thresholds are QUANTILES of the scores (5%..95%), so the chosen threshold always
    predicts both classes, however compressed the probabilities are (fixes 'predict all 1')."""
    p1 = probs[:, 1]
    ts = np.unique(np.quantile(p1, np.linspace(0.05, 0.95, 91)))
    f1s = np.array([f1_score(labels, (p1 >= t).astype(int), average='macro', zero_division=0) for t in ts])
    k = min(5, len(f1s))
    sm = np.convolve(np.pad(f1s, (k // 2, k // 2), mode='edge'), np.ones(k) / k, mode='valid')[:len(f1s)]
    i = int(np.argmax(sm))
    default_f1 = f1_score(labels, (p1 >= 0.5).astype(int), average='macro', zero_division=0)
    return float(ts[i]), float(f1s[i]), float(default_f1)

def safe_auc(y, p):
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def dev_score(f1, auc):
    return f1 + 0.1 * (0.0 if math.isnan(auc) else auc)     # AUC term makes selection less noisy than F1 alone

# ---------------------------- Check 1: linear probe ---------------------------
def probe_check1(Xtr, Mtr, ytr, Xdv, Mdv, ydv, name, n=12000):
    t0 = time.time()
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(ytr), min(n, len(ytr)), replace=False)
    def views(X, M):
        Xf, Mf = X.float(), M.float().unsqueeze(-1)
        mean = (Xf * Mf).sum(1) / Mf.sum(1).clamp(min=1)
        last = Xf[torch.arange(X.size(0)), (M.sum(1) - 1).clamp(min=0)]
        return mean.numpy(), last.numpy()
    tr_mean, tr_last = views(Xtr[idx], Mtr[idx]); dv_mean, dv_last = views(Xdv, Mdv)
    res = {}
    for vname, a, b in (('all-chunk mean', tr_mean, dv_mean), ('last chunk', tr_last, dv_last)):
        sc = StandardScaler().fit(a)
        clf = LogisticRegression(C=0.05, max_iter=2000, class_weight='balanced').fit(sc.transform(a), ytr[idx])
        p = clf.predict_proba(sc.transform(b))
        res[vname] = (safe_auc(ydv, p[:, 1]), calibrate_threshold(p, ydv)[1])
    print(f"[{name}] CHECK 1 (logistic regression on frozen features, {len(idx)} train docs, {time.time()-t0:.0f}s): " +
          " | ".join(f"{k}: Dev AUC {v[0]:.3f}, F1 {v[1]:.3f}" for k, v in res.items()))
    best_auc = max(v[0] for v in res.values())
    if best_auc < 0.55:
        print("  -> frozen features carry almost no label signal (cause 1). Stage B (adapters / unfrozen layers) has to create it.")
    else:
        print("  -> frozen features DO carry signal (cause 2 was the culprit). Stage A below should reach at least this level.")
    return res

# ---------------------------- Stage A: head on cached features ----------------
def train_head(head: nn.Module, Xtr, Mtr, ytr, Xdv, Mdv, ydv, name):
    head.to(device)
    opt = AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WD)
    n = len(ytr); steps_per = math.ceil(n / LP_BS); total = LP_EPOCHS * steps_per; warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    yt = torch.as_tensor(ytr)
    best, best_state, bad, t0 = -1.0, None, 0, time.time()
    for ep in range(1, LP_EPOCHS + 1):
        head.train(); perm = torch.randperm(n); tot = 0.0
        for s in range(0, n, LP_BS):
            b = perm[s:s + LP_BS]
            logits, _ = head(Xtr[b].to(device).float(), Mtr[b].to(device))
            loss = crit(logits, yt[b].to(device))
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item() * len(b)
        p = softmax_np(head_predict(head, Xdv, Mdv)[0])
        tau, f1, f1d = calibrate_threshold(p, ydv); auc = safe_auc(ydv, p[:, 1]); sc = dev_score(f1, auc)
        flag = ''
        if sc > best:
            best, bad, flag = sc, 0, ' *'
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
        print(f"  [{name} Stage A] ep {ep:2d} | train loss {tot/n:.4f} | Dev F1@tau* {f1:.4f} (F1@0.5 {f1d:.4f}) | AUC {auc:.4f}{flag}")
        if bad >= LP_PATIENCE:
            print(f"  [{name} Stage A] early stop"); break
    head.load_state_dict(best_state)
    p = softmax_np(head_predict(head, Xdv, Mdv)[0])
    tau, f1, _ = calibrate_threshold(p, ydv)
    print(f"✓ [{name} Stage A] best Dev Macro-F1 {f1:.4f} @tau*={tau:.3f}, AUC {safe_auc(ydv, p[:, 1]):.4f} ({(time.time()-t0)/60:.1f} min)")
    return p

# ---------------------------- Stage B: PEFT fine-tuning ------------------------
def peft_groups(model: nn.Module, lr_head=FT_LR_HEAD, lr_peft=FT_LR_PEFT, e_mult: float = 1.0):
    head, peft, lam = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith('head.'): head.append(p)
        elif 'lora_E' in n: lam.append(p)
        else: peft.append(p)
    groups = [{'params': head, 'lr': lr_head, 'weight_decay': 0.01, 'name': 'head'}]
    if peft: groups.append({'params': peft, 'lr': lr_peft, 'weight_decay': 0.0, 'name': 'peft'})
    if lam:  groups.append({'params': lam, 'lr': lr_peft * e_mult, 'weight_decay': 0.0, 'name': 'lambda'})
    return groups

def make_ce_step(criterion):
    def step(model, batch):
        logits = model(batch['input_ids'], batch['mask'])
        return criterion(logits.float(), batch['label'].view(1))
    return step

def finetune(model: DocModel, name: str, run_dir: str, param_groups, step_fn,
             doc_pool=None, after_step=None, minutes=None):
    minutes = FT_MINUTES if minutes is None else minutes
    os.makedirs(run_dir, exist_ok=True)
    if minutes <= 0:
        print(f"[{name} Stage B] skipped (FT_MINUTES=0)"); return
    doc_pool = np.arange(len(train_store)) if doc_pool is None else np.asarray(doc_pool)
    opt = AdamW(param_groups)
    for g in opt.param_groups: g['base_lr'] = g['lr']
    params = [p for g in param_groups for p in g['params']]
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    n_tr, n_all = sum(p.numel() for p in params), sum(p.numel() for p in model.parameters())
    print(f"[{name} Stage B] trainable {n_tr/1e6:.2f}M / {n_all/1e6:.1f}M params ({100*n_tr/n_all:.2f}%) | "
          f"budget {minutes} min | {FT_ACCUM} docs/step | groups: " + ", ".join(f"{g.get('name','?')}={g['lr']:.1e}" for g in param_groups))
    def snapshot():
        return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in trainable or k.endswith('rank_mask')}
    def evaluate():
        p = softmax_np(predict_store(model, dev_store)[0])
        tau, f1, f1d = calibrate_threshold(p, dev_labels); auc = safe_auc(dev_labels, p[:, 1])
        return dev_score(f1, auc), f1, f1d, auc, tau, float((p[:, 1] >= tau).mean())
    sc, f1, f1d, auc, tau, pr = evaluate()
    print(f"[{name} Stage B] start (= Stage-A head on live features): Dev F1@tau* {f1:.4f} | AUC {auc:.4f}")
    best, best_state, bad = sc, snapshot(), 0
    scaler = make_scaler()
    budget, t0 = minutes * 60, time.time()
    WARM = 20
    def set_lr(step):
        frac = min(1.0, (time.time() - t0) / budget)
        s = min(1.0, (step + 1) / WARM) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))   # time-based cosine
        for g in opt.param_groups: g['lr'] = g['base_lr'] * s
    rng = np.random.RandomState(SEED + 1)
    order, pos, docs, step, skipped, run_loss = rng.permutation(doc_pool), 0, 0, 0, 0, []
    model.train(); set_lr(0); opt.zero_grad(set_to_none=True)
    while time.time() - t0 < budget:
        if pos >= len(order): order, pos = rng.permutation(doc_pool), 0
        i = int(order[pos]); pos += 1
        ids, mask = train_store.get(i)
        batch = {'idx': i, 'input_ids': ids.to(device), 'mask': mask.to(device),
                 'label': torch.tensor(int(train_store.labels[i]), device=device)}
        with amp_ctx():
            loss = step_fn(model, batch)
        if not torch.isfinite(loss):
            skipped += 1; continue
        scaler.scale(loss / FT_ACCUM).backward()
        docs += 1; run_loss.append(loss.item())
        if docs % FT_ACCUM == 0:
            scaler.unscale_(opt)
            if step == 0:   # diagnostic: gradients must reach every parameter group
                norms = []
                for g in opt.param_groups:
                    gs = [p.grad.float().norm() for p in g['params'] if p.grad is not None]
                    norms.append(f"{g.get('name','?')}={torch.stack(gs).norm().item():.2e}" if gs else f"{g.get('name','?')}=NO GRAD")
                print(f"  [{name} Stage B] first-step grad norms: " + ", ".join(norms))
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            if after_step is not None: after_step(min(1.0, (time.time() - t0) / budget))
            opt.zero_grad(set_to_none=True); step += 1; set_lr(step)
        if docs > 0 and docs % FT_EVAL_EVERY == 0:
            sc, f1, f1d, auc, tau, pr = evaluate(); model.train()
            flag = ''
            if sc > best: best, best_state, bad, flag = sc, snapshot(), 0, ' *'
            else: bad += 1
            print(f"  [{name} Stage B] {docs} docs / {step} steps | loss {np.mean(run_loss[-FT_EVAL_EVERY:]):.4f} | "
                  f"Dev F1@tau* {f1:.4f} (F1@0.5 {f1d:.4f}) | AUC {auc:.4f} | pos-rate {pr:.2f} | "
                  f"{(time.time()-t0)/60:.0f}/{minutes} min{flag}")
            if bad >= FT_PATIENCE:
                print(f"  [{name} Stage B] early stop"); break
    if docs % FT_EVAL_EVERY != 0:
        sc, f1, f1d, auc, tau, pr = evaluate()
        if sc > best: best, best_state = sc, snapshot()
        print(f"  [{name} Stage B] end: {docs} docs | Dev F1@tau* {f1:.4f} | AUC {auc:.4f}")
    if skipped: print(f"  ⚠ {skipped} documents gave a non-finite loss and were skipped")
    model.load_state_dict(best_state, strict=False)
    torch.save(best_state, os.path.join(run_dir, "best_trainable.pt"))

# ---------------------------- packaging / persistence -------------------------
def package_submission(test_probs: np.ndarray, ids: List[str], threshold: float, output_dir: str, zip_filename: str):
    os.makedirs(output_dir, exist_ok=True)
    preds = (test_probs[:, 1] >= threshold).astype(int)
    json_path = os.path.join(output_dir, "prediction.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump([{"id": cid, "pred": int(p)} for cid, p in zip(ids, preds)], f, indent=2)
    zip_path = os.path.join(WORK_DIR, zip_filename)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        verified = json.load(zf.open("prediction.json"))
    assert len(verified) == len(test_records), "Mismatch in test count!"
    c1 = sum(r['pred'] for r in verified); c0 = len(verified) - c1
    print(f"✓ {zip_path} | {len(verified)} predictions -> Class 0: {c0} ({c0/len(verified)*100:.1f}%), Class 1: {c1} ({c1/len(verified)*100:.1f}%)")
    return zip_path

def save_member(key: str, label: str, dev_probs: np.ndarray, test_probs: np.ndarray, extra: Optional[dict] = None):
    tau, f1, f1d = calibrate_threshold(dev_probs, dev_labels)
    RUN_PREDICTIONS_DEV[key], RUN_PREDICTIONS_TEST[key] = dev_probs, test_probs
    RUN_METRICS_SUMMARY[label] = {'f1': f1, 'threshold': tau}
    np.save(os.path.join(OUT_DIR, f"{key}_dev.npy"), dev_probs)
    np.save(os.path.join(OUT_DIR, f"{key}_test.npy"), test_probs)
    meta = {'key': key, 'label': label, 'f1': f1, 'threshold': tau, 'auc': safe_auc(dev_labels, dev_probs[:, 1])}
    meta.update(extra or {})
    with open(os.path.join(OUT_DIR, f"{key}_meta.json"), 'w') as f: json.dump(meta, f, indent=2)
    return tau, f1, f1d

def finalize_run(model: DocModel, key: str, label: str, run_dir: str, zip_name: str, stage_a_f1=None):
    dl, tl = predict_store(model, dev_store, 'Dev')[0], predict_store(model, test_store, 'Test')[0]
    dev_probs, test_probs = softmax_np(dl), softmax_np(tl)
    tau, f1, f1d = save_member(key, label, dev_probs, test_probs, {'stage_a_f1': stage_a_f1})
    print(f"{label} | tau*={tau:.3f} | Dev Macro-F1 {f1:.4f} (F1@0.5 {f1d:.4f}, AUC {safe_auc(dev_labels, dev_probs[:, 1]):.4f})")
    print(classification_report(dev_labels, (dev_probs[:, 1] >= tau).astype(int), digits=4, zero_division=0))
    package_submission(test_probs, test_ids, tau, run_dir, zip_name)

def run_lpft(key, label, run_dir, zip_name, encoder: FeatureEncoder, head: nn.Module, make_groups,
             step_fn=None, doc_pool=None, after_step=None):
    """Stage A (head on cached frozen features) -> Stage B (PEFT fine-tuning) -> Dev/Test predictions."""
    os.makedirs(run_dir, exist_ok=True)
    encoder.to(device)
    tag = f"{key}_{CHUNK_LEN}_{HEAD_CHUNKS}_{TAIL_CHUNKS}_{len(train_store)}"
    Xtr, Mtr = extract_features(encoder, train_store, f'{key} train', cache_key=f"{tag}_train")
    Xdv, Mdv = extract_features(encoder, dev_store, f'{key} dev', cache_key=f"{tag}_dev")
    probe_check1(Xtr, Mtr, train_store.labels, Xdv, Mdv, dev_labels, key)
    pA = train_head(head, Xtr, Mtr, train_store.labels, Xdv, Mdv, dev_labels, key)
    stage_a_f1 = calibrate_threshold(pA, dev_labels)[1]
    del Xtr, Mtr; free_mem()
    model = DocModel(encoder, head).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    finetune(model, key, run_dir, make_groups(model), step_fn or make_ce_step(criterion),
             doc_pool=doc_pool, after_step=after_step)
    finalize_run(model, key, label, run_dir, zip_name, stage_a_f1=stage_a_f1)
    return model

## RUN 1: Architecture 4 + LORMA  [FLAGSHIP]
Backbone `ALiBi + SwiGLU + RMSNorm` (~108M, Phase-1 PPL 23.66) · LORMA r=16 attention / r=8 FFN · BiLSTM document head

In [ ]:
# RUN 1: Architecture 4 (ALiBi + SwiGLU + RMSNorm) + LORMA + hierarchical BiLSTM head  [FLAGSHIP]
#   Stage A: frozen backbone -> chunk vectors cached once -> BiLSTM head trained for many epochs
#   Stage B: LORMA adapters (B = 0, so the model starts exactly at the Stage-A solution) + head, jointly
class RMSNormArch4(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]
    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))
    closest_pow2 = 2 ** math.floor(math.log2(n_heads))
    slopes_a = get_slopes_power_of_2(closest_pow2)
    slopes_b = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
    return torch.tensor(slopes_a + slopes_b)

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNormArch4(self.head_dim)
        self.k_norm = RMSNormArch4(self.head_dim)
        self.register_buffer("slopes", get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        k = self.k_norm(self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        pos_q = torch.arange(S, device=x.device).unsqueeze(1)
        pos_k = torch.arange(S, device=x.device).unsqueeze(0)
        alibi_bias = self.slopes * (pos_k - pos_q).unsqueeze(0)
        causal_mask = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = (alibi_bias + causal_mask.unsqueeze(0).unsqueeze(0)).to(q.dtype)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, S, self.d_model))

class SwiGLUArch4(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class TransformerBlockArch4(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNormArch4(d_model)
        self.attn  = ALiBiCausalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNormArch4(d_model)
        self.ffn   = SwiGLUArch4(d_model, d_ff, dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, vocab_size: int = 8000, d_model: int = D_MODEL, n_layers: int = N_LAYERS,
                 n_heads: int = N_HEADS, d_ff: int = FF_SWIGLU, dropout: float = 0.1):
        super().__init__()
        self.vocab_size, self.d_model = vocab_size, d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlockArch4(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNormArch4(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.tok_emb(idx))
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x)

# ---------------------------------- LORMA -------------------------------------
class LORMALinear(nn.Module):
    """y = W0 x + s * (alpha/r) * B A x. B = 0 at init, so injecting the adapter does not change the
    pretrained model (the Stage-A head stays valid). A learnable magnitude s (init 1) rescales the update."""
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer, self.rank, self.scaling = base_layer, rank, alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        self.lora_s = nn.Parameter(torch.ones(1))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        for p in self.base_layer.parameters(): p.requires_grad = False
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        delta = (self.dropout(x) @ self.lora_A.t().to(x.dtype)) @ self.lora_B.t().to(x.dtype)
        return self.base_layer(x) + delta * (self.lora_s.to(x.dtype) * self.scaling)

def inject_lorma(model: nn.Module, r_attn: int = 16, r_ffn: int = 8, alpha: float = 32.0, dropout: float = 0.05,
                 attn_names=('q_proj', 'k_proj', 'v_proj', 'out_proj'), ffn_names=('gate_proj', 'up_proj', 'down_proj')):
    for p in model.parameters(): p.requires_grad = False
    count = 0
    for block in model.blocks:
        for name in attn_names:
            l = getattr(block.attn, name, None)
            if isinstance(l, nn.Linear):
                setattr(block.attn, name, LORMALinear(l, rank=r_attn, alpha=alpha, dropout=dropout)); count += 1
        for name in ffn_names:
            l = getattr(block.ffn, name, None)
            if isinstance(l, nn.Linear):
                setattr(block.ffn, name, LORMALinear(l, rank=r_ffn, alpha=alpha, dropout=dropout)); count += 1
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Injected LORMA into {count} linear projections (r_attn={r_attn}, r_ffn={r_ffn}) | {n/1e6:.2f}M adapter params")

# ----------------------------- build / train ----------------------------------
if 1 in RUNS_TO_EXECUTE:
    free_mem(); AMP_ON = True
    RUN1_DIR = os.path.join(WORK_DIR, "run1_arch4_lorma")
    backbone4 = ALiBiSwiGLULM(vocab_size=tokenizer.vocab_size)
    load_pretrained(backbone4, ARCH4_CHECKPOINT, "Arch4 backbone")
    backbone4 = backbone4.to(device)
    disable_dropout(backbone4)
    backbone_sanity(backbone4, "Arch4")
    inject_lorma(backbone4, r_attn=16, r_ffn=8, alpha=32.0, dropout=0.05)
    backbone4.to(device)
    enc4 = FeatureEncoder(backbone4)
    head1 = BiLSTMDocHead(enc4.out_dim)
    model1 = run_lpft('Run1_Arch4_LORMA', 'Run 1 (Arch 4 LORMA)', RUN1_DIR, "submission_run1_arch4_lorma_BEST.zip",
                      enc4, head1, lambda m: peft_groups(m))

    # Teacher outputs for Run 4 (knowledge distillation) on a fixed random subset of training docs
    print(f"Caching teacher outputs on {min(KD_DOCS, len(train_store))} training documents for Run 4 (KD)...")
    kd_idx = np.sort(np.random.RandomState(SEED).choice(len(train_store), min(KD_DOCS, len(train_store)), replace=False))
    t_logits, t_feat = predict_store(model1, train_store.subset(kd_idx), 'teacher')
    torch.save({'idx': kd_idx, 'logits': t_logits, 'feat': t_feat, 'n_train': len(train_store)},
               os.path.join(OUT_DIR, "teacher_kd.pt"))
    print(f"✓ Teacher cache saved: logits {tuple(t_logits.shape)}, feats {tuple(t_feat.shape)}")
    del model1, enc4, head1, backbone4, t_logits, t_feat
    free_mem()

## RUN 2: Architecture 3 + AdaLoRA
Backbone `GQA + learned positions + LayerNorm + GELU` · AdaLoRA (SVD form, rank pruning) · gated-attention chunk aggregator

In [ ]:
# RUN 2: Architecture 3 (GQA + learned positions + LayerNorm + GELU) + AdaLoRA + gated-attention head
#   AdaLoRA: W = W0 + (alpha/r) P diag(Lambda * mask) Q, Lambda = 0 at init (identity), orthogonality
#   regulariser on P and Q, importance-based rank pruning with a cubic budget schedule over the run.
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, n_kv_heads: int = N_KV_HEADS, dropout: float = 0.1):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        k = torch.repeat_interleave(k, self.num_queries_per_kv, dim=1)
        v = torch.repeat_interleave(v, self.num_queries_per_kv, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, self.d_model))

class FeedForwardArch3(nn.Module):
    def __init__(self, d_model: int = D_MODEL, d_ff: int = FF_GELU, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff); self.act = nn.GELU()
        self.fc2 = nn.Linear(d_ff, d_model); self.drop = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.fc2(self.act(self.fc1(x))))

class GQABlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, n_heads, n_kv_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardArch3(d_model, d_ff, dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))

class GQADecoderLM(nn.Module):
    def __init__(self, vocab_size: int = 8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 n_kv_heads=N_KV_HEADS, d_ff=FF_GELU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GQABlock(d_model, n_heads, n_kv_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

# -------------------------------- AdaLoRA -------------------------------------
class AdaLoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer, self.r, self.scaling = base_layer, r, alpha / r
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.lora_P = nn.Parameter(torch.empty(base_layer.out_features, r))
        self.lora_E = nn.Parameter(torch.zeros(r))
        self.lora_Q = nn.Parameter(torch.empty(r, base_layer.in_features))
        nn.init.orthogonal_(self.lora_P); nn.init.orthogonal_(self.lora_Q)
        self.register_buffer('rank_mask', torch.ones(r))
        self.register_buffer('importance', torch.zeros(r))
        for p in self.base_layer.parameters(): p.requires_grad = False
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lam = (self.lora_E * self.rank_mask).to(x.dtype)
        h = (self.dropout(x) @ self.lora_Q.t().to(x.dtype)) * lam
        return self.base_layer(x) + (h @ self.lora_P.t().to(x.dtype)) * self.scaling
    def get_ortho_loss(self):
        P, Q = self.lora_P.float(), self.lora_Q.float()
        eye = torch.eye(self.r, device=P.device)
        return torch.norm(P.t() @ P - eye, p='fro') + torch.norm(Q @ Q.t() - eye, p='fro')

def inject_adalora(model: nn.Module, r: int = 16, alpha: float = 32.0):
    for p in model.parameters(): p.requires_grad = False
    count = 0
    for block in model.blocks:
        for name in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            l = getattr(block.attn, name)
            if isinstance(l, nn.Linear):
                setattr(block.attn, name, AdaLoRALinear(l, r=r, alpha=alpha)); count += 1
        for name in ('fc1', 'fc2'):
            l = getattr(block.ffn, name)
            if isinstance(l, nn.Linear):
                setattr(block.ffn, name, AdaLoRALinear(l, r=r, alpha=alpha)); count += 1
    print(f"✓ Injected AdaLoRA into {count} linear projections.")

class AdaLoRAController:
    """Importance s_i = EMA(|Lambda_i * dL/dLambda_i|); the global top-b triplets are kept. The budget b
    follows a cubic schedule in the fraction of the time budget used (full rank -> final_ratio)."""
    def __init__(self, model, final_ratio=0.5, t_init=0.15, t_final=0.30, every=10, beta=0.85):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.full = sum(m.r for m in self.mods)
        self.target = int(self.full * final_ratio)
        self.t_init, self.t_final, self.every, self.beta, self.calls = t_init, t_final, every, beta, 0
    def budget(self, frac):
        if frac < self.t_init: return self.full
        if frac >= 1 - self.t_final: return self.target
        f = (frac - self.t_init) / max(1e-6, 1 - self.t_init - self.t_final)
        return int(self.target + (self.full - self.target) * (1 - f) ** 3)
    @torch.no_grad()
    def __call__(self, frac):
        for m in self.mods:
            if m.lora_E.grad is not None:
                s = (m.lora_E * m.lora_E.grad).abs()
                if torch.isfinite(s).all():
                    m.importance.mul_(self.beta).add_((1 - self.beta) * s)
        self.calls += 1
        if self.calls % self.every: return
        b = self.budget(frac)
        if b >= self.full: return
        scores = torch.cat([m.importance * m.rank_mask for m in self.mods])
        thr = torch.topk(scores, b).values[-1]
        for m in self.mods:
            m.rank_mask.copy_(((m.importance * m.rank_mask) >= thr).float() * m.rank_mask)
    def kept(self):
        return int(sum(m.rank_mask.sum().item() for m in self.mods))

ORTHO_GAMMA = 0.1

if 2 in RUNS_TO_EXECUTE:
    free_mem(); AMP_ON = True
    RUN2_DIR = os.path.join(WORK_DIR, "run2_arch3_adalora")
    backbone3 = GQADecoderLM(vocab_size=tokenizer.vocab_size)
    load_pretrained(backbone3, ARCH3_CHECKPOINT, "Arch3 backbone")
    backbone3 = backbone3.to(device)
    disable_dropout(backbone3)
    backbone_sanity(backbone3, "Arch3")
    inject_adalora(backbone3, r=16, alpha=32.0)
    backbone3.to(device)
    enc3 = FeatureEncoder(backbone3)
    head2 = GatedAttnDocHead(enc3.out_dim)
    ada_ctrl = AdaLoRAController(enc3)
    ada_mods = ada_ctrl.mods
    crit2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    def adalora_step(model, batch):
        logits = model(batch['input_ids'], batch['mask'])
        ce = crit2(logits.float(), batch['label'].view(1))
        ortho = sum(m.get_ortho_loss() for m in ada_mods) / len(ada_mods)
        return ce + ORTHO_GAMMA * ortho
    model2 = run_lpft('Run2_Arch3_AdaLoRA', 'Run 2 (Arch 3 AdaLoRA)', RUN2_DIR, "submission_run2_arch3_adalora.zip",
                      enc3, head2, lambda m: peft_groups(m, e_mult=4.0), step_fn=adalora_step, after_step=ada_ctrl)
    print(f"AdaLoRA final rank budget: {ada_ctrl.kept()} / {ada_ctrl.full} singular triplets kept")
    del model2, enc3, head2, backbone3, ada_ctrl, ada_mods
    free_mem()

## RUN 3: Architecture 2 + layer-wise discriminative fine-tuning
Backbone `RoPE + SwiGLU + RMSNorm` · top 4 blocks unfrozen, LR decays by ξ = 0.85 per block · temporal-conv head

In [ ]:
# RUN 3: Architecture 2 (RoPE + SwiGLU + RMSNorm) + layer-wise discriminative fine-tuning + temporal-conv head
#   Top 4 blocks + final norm are unfrozen; LR(block at depth d from the top) = LR_LAYER_TOP * XI^d.
class RMSNormArch2(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.type_as(x) * self.weight.type_as(x)

class RotaryEmbeddingArch2(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = MAX_POS, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len, dtype=torch.float)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())
    def forward(self, x: torch.Tensor, seq_len: int):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(1).to(q.dtype)
    sin = sin.unsqueeze(0).unsqueeze(1).to(q.dtype)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RoPECausalAttention(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, dropout: float = 0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNormArch2(self.head_dim)
        self.k_norm = RMSNormArch2(self.head_dim)
    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        k = self.k_norm(self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, S, self.d_model))

class SwiGLUArch2(nn.Module):
    def __init__(self, d_model: int = D_MODEL, d_ff: int = FF_SWIGLU, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class RoPEBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=FF_SWIGLU, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNormArch2(d_model)
        self.attn  = RoPECausalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNormArch2(d_model)
        self.ffn   = SwiGLUArch2(d_model, d_ff, dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.ffn(self.norm2(x))

class RoPESwiGLUDecoderLM(nn.Module):
    def __init__(self, vocab_size: int = 8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 d_ff=FF_SWIGLU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbeddingArch2(d_model // n_heads, max_seq_len=max_seq_len)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([RoPEBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNormArch2(d_model)
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, S = idx.shape
        cos, sin = self.rope(idx, S)
        x = self.drop(self.tok_emb(idx))
        for block in self.blocks: x = block(x, cos, sin)
        return self.norm_f(x)

N_UNFROZEN = 4

if 3 in RUNS_TO_EXECUTE:
    free_mem(); AMP_ON = True
    RUN3_DIR = os.path.join(WORK_DIR, "run3_arch2_layerwise")
    backbone2 = RoPESwiGLUDecoderLM(vocab_size=tokenizer.vocab_size)
    load_pretrained(backbone2, ARCH2_CHECKPOINT, "Arch2 backbone")
    backbone2 = backbone2.to(device)
    disable_dropout(backbone2)
    backbone_sanity(backbone2, "Arch2")
    for p in backbone2.parameters(): p.requires_grad = False
    for block in backbone2.blocks[-N_UNFROZEN:]:
        for p in block.parameters(): p.requires_grad = True
    for p in backbone2.norm_f.parameters(): p.requires_grad = True
    enc2 = FeatureEncoder(backbone2)
    head3 = TemporalConvDocHead(enc2.out_dim)

    def layerwise_groups(model):
        bb = model.encoder.backbone
        groups = [{'params': list(model.head.parameters()), 'lr': FT_LR_HEAD, 'weight_decay': 0.01, 'name': 'head'},
                  {'params': list(bb.norm_f.parameters()), 'lr': LR_LAYER_TOP, 'weight_decay': 0.0, 'name': 'norm_f'}]
        for d in range(N_UNFROZEN):
            blk = bb.blocks[len(bb.blocks) - 1 - d]
            groups.append({'params': list(blk.parameters()), 'lr': LR_LAYER_TOP * (XI ** d), 'weight_decay': 0.01,
                           'name': f'block{len(bb.blocks) - d}'})
        print("Layer-wise LRs (top -> bottom):", [f"{g['lr']:.2e}" for g in groups[2:]])
        return groups

    model3 = run_lpft('Run3_Arch2_Layerwise', 'Run 3 (Arch 2 Layerwise)', RUN3_DIR, "submission_run3_arch2_layerwise.zip",
                      enc2, head3, layerwise_groups)
    del model3, enc2, head3, backbone2
    free_mem()

## RUN 4: Knowledge distillation (Run-1 teacher → Arch 1 student + LoRA)

In [ ]:
# RUN 4: Knowledge distillation  (fine-tuned Run-1 model = teacher  ->  Arch 1 + LoRA = student)
#   Teacher : Run-1 model, frozen; its logits and document features on KD_DOCS training docs were
#   Student : Arch 1 (sinusoidal + LayerNorm + GELU) + LoRA (attn r=16, FFN r=8) + the same BiLSTM head.
#   Stage A : student head on cached frozen student features (task loss).
#   Stage B : 0.60 * KD(T=3) + 0.30 * task CE + 0.10 * (1 - cos(student feat, teacher feat)).
def sinusoidal_pos_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class CausalSelfAttentionArch1(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, dropout: float = 0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, self.d_model))

class FeedForwardArch1(nn.Module):
    def __init__(self, d_model: int = D_MODEL, d_ff: int = FF_GELU, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.net(x)

class BlockArch1(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttentionArch1(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardArch1(d_model, d_ff, dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))

class StudentArch1LM(nn.Module):
    def __init__(self, vocab_size: int = 8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 d_ff=FF_GELU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", sinusoidal_pos_encoding(max_seq_len, d_model).unsqueeze(0))
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([BlockArch1(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pe[:, :T].to(self.tok_emb.weight.dtype))
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

T_KD = 3.0

def find_teacher_file():
    for d in [OUT_DIR] + list(PREV_OUT_DIRS):
        p = os.path.join(d, "teacher_kd.pt")
        if os.path.exists(p): return p
    hits = glob.glob("/kaggle/input/**/teacher_kd.pt", recursive=True)
    return hits[0] if hits else None

if 4 in RUNS_TO_EXECUTE:
    free_mem(); AMP_ON = True
    RUN4_DIR = os.path.join(WORK_DIR, "run4_kd_arch4_to_arch1")
    tpath = find_teacher_file()
    assert tpath is not None, "teacher_kd.pt not found: run Run 1 first (or add its output via PREV_OUT_DIRS)."
    TEACH = torch.load(tpath, weights_only=False)
    assert TEACH['n_train'] == len(train_store), "teacher cache was made with a different training set (DRY_RUN?)"
    KD_ROW = {int(i): r for r, i in enumerate(TEACH['idx'])}
    T_LOGITS, T_FEAT = TEACH['logits'].float(), TEACH['feat'].float()
    print(f"✓ Teacher cache loaded from {tpath}: {len(KD_ROW)} documents")

    student_bb = StudentArch1LM(vocab_size=tokenizer.vocab_size)
    load_pretrained(student_bb, ARCH1_CHECKPOINT, "Arch1 student backbone")
    student_bb = student_bb.to(device)
    disable_dropout(student_bb)
    backbone_sanity(student_bb, "Arch1")
    for p in student_bb.parameters(): p.requires_grad = False
    for block in student_bb.blocks:
        for name in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            setattr(block.attn, name, LORMALinear(getattr(block.attn, name), rank=16, alpha=32.0, dropout=0.05))
        for j in (0, 2):
            block.ffn.net[j] = LORMALinear(block.ffn.net[j], rank=8, alpha=32.0, dropout=0.05)
    student_bb.to(device)
    enc1 = FeatureEncoder(student_bb)
    head4 = BiLSTMDocHead(enc1.out_dim)          # same head as the teacher -> same feature size for the rep. loss

    task_loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    def kd_step(model, batch):
        s_logits, s_feat = model(batch['input_ids'], batch['mask'], return_feat=True)
        r = KD_ROW[int(batch['idx'])]
        t_logits = T_LOGITS[r:r + 1].to(device); t_feat = T_FEAT[r:r + 1].to(device)
        s_logits, s_feat = s_logits.float(), s_feat.float()
        loss_kd = F.kl_div(F.log_softmax(s_logits / T_KD, -1), F.softmax(t_logits / T_KD, -1), reduction='batchmean') * T_KD ** 2
        loss_task = task_loss_fn(s_logits, batch['label'].view(1))
        loss_rep = 1.0 - F.cosine_similarity(s_feat, t_feat).mean()
        return 0.60 * loss_kd + 0.30 * loss_task + 0.10 * loss_rep

    student_model = run_lpft('Run4_KD_Arch4_to_Arch1', 'Run 4 (KD Arch 4 -> Arch 1)', RUN4_DIR,
                             "submission_run4_kd_arch4_to_arch1.zip", enc1, head4, lambda m: peft_groups(m),
                             step_fn=kd_step, doc_pool=TEACH['idx'])
    del student_model, enc1, head4, student_bb, TEACH, T_LOGITS, T_FEAT
    free_mem()

## RUN 5: Ensemble

In [ ]:
# (OPTIONAL) Classical TF-IDF + logistic regression as an extra ensemble member.
# Off by default: the assignment is about adapting the Phase-1 LMs. Turn on USE_TFIDF_MODEL only if
# the rules allow a non-LM component in the submission. Runs in ~5 min on CPU.
if USE_TFIDF_MODEL and 5 in RUNS_TO_EXECUTE:
    from sklearn.feature_extraction.text import TfidfVectorizer
    t0 = time.time()
    vec = TfidfVectorizer(max_features=300000, ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_df=0.9, dtype=np.float32)
    Xtr_t = vec.fit_transform([r['text'] for r in train_records]); ytr_t = np.array([r['label'] for r in train_records])
    clf_t = LogisticRegression(C=4.0, class_weight='balanced', max_iter=3000).fit(Xtr_t, ytr_t)
    dp = clf_t.predict_proba(vec.transform([r['text'] for r in dev_records]))
    tp = clf_t.predict_proba(vec.transform([r['text'] for r in test_records]))
    t_, f_, _ = save_member('Aux_TFIDF_LR', 'Aux (TF-IDF + LR)', dp, tp)
    print(f"TF-IDF+LR | tau*={t_:.3f} | Dev Macro-F1 {f_:.4f} | AUC {safe_auc(dev_labels, dp[:, 1]):.4f} ({time.time()-t0:.0f}s)")
    del Xtr_t, clf_t, vec
else:
    print("TF-IDF auxiliary model disabled (USE_TFIDF_MODEL=False).")

In [ ]:
# RUN 5: Multi-model ensemble & final submission
#   Candidates (each with its own calibrated tau on Dev): equal weights, Dev-F1 weights, and
#   logistic-regression stacking scored with 5-fold out-of-fold Dev predictions (honest estimate).
#   Stacking must beat the simple blends by > 0.01; the best single model is used if nothing beats it.
from sklearn.model_selection import StratifiedKFold

if 5 in RUNS_TO_EXECUTE:
    ENSEMBLE_DIR = os.path.join(WORK_DIR, "master_ensemble"); os.makedirs(ENSEMBLE_DIR, exist_ok=True)
    MEMBERS = ['Run1_Arch4_LORMA', 'Run2_Arch3_AdaLoRA', 'Run3_Arch2_Layerwise', 'Run4_KD_Arch4_to_Arch1', 'Aux_TFIDF_LR']
    for k in MEMBERS:
        if k in RUN_PREDICTIONS_DEV: continue
        for d in [OUT_DIR] + list(PREV_OUT_DIRS):
            pd_, pt_ = os.path.join(d, f"{k}_dev.npy"), os.path.join(d, f"{k}_test.npy")
            if os.path.exists(pd_) and os.path.exists(pt_):
                dv, ts = np.load(pd_), np.load(pt_)
                if len(dv) == len(dev_labels) and len(ts) == len(test_ids):
                    RUN_PREDICTIONS_DEV[k], RUN_PREDICTIONS_TEST[k] = dv, ts
                    if k == 'Aux_TFIDF_LR' and not USE_TFIDF_MODEL:
                        del RUN_PREDICTIONS_DEV[k], RUN_PREDICTIONS_TEST[k]
                    break
    names = [k for k in MEMBERS if k in RUN_PREDICTIONS_DEV]
    assert names, "No run predictions found. Execute Runs 1-4 first (or point PREV_OUT_DIRS at their outputs)."
    member_f1 = {k: calibrate_threshold(RUN_PREDICTIONS_DEV[k], dev_labels)[1] for k in names}
    print("Members:", {k: round(v, 4) for k, v in member_f1.items()})

    def _norm(w): s = sum(w.values()); return {k: v / s for k, v in w.items()}
    def _blend(w, src): return sum(w[k] * src[k] for k in w)
    cands = {'equal': _norm({k: 1.0 for k in names}),
             'f1-weighted': _norm({k: max(member_f1[k] - 0.45, 0.01) for k in names})}
    results = {}
    for cname, w in cands.items():
        dp = _blend(w, RUN_PREDICTIONS_DEV); t, f1o, f1d = calibrate_threshold(dp, dev_labels)
        results[cname] = {'t': t, 'f1': f1o, 'test': _blend(w, RUN_PREDICTIONS_TEST)}
        print(f"  {cname:12s}: Dev Macro-F1 {f1o:.4f} @tau*={t:.3f} | AUC {safe_auc(dev_labels, dp[:, 1]):.4f}")

    def _logit(p): return np.log(np.clip(p[:, 1], 1e-6, 1 - 1e-6) / np.clip(p[:, 0], 1e-6, 1 - 1e-6))
    if len(names) >= 2:
        Xd = np.stack([_logit(RUN_PREDICTIONS_DEV[k]) for k in names], 1)
        Xt = np.stack([_logit(RUN_PREDICTIONS_TEST[k]) for k in names], 1)
        oof = np.zeros(len(dev_labels))
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xd, dev_labels):
            oof[va] = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000).fit(Xd[tr], dev_labels[tr]).predict_proba(Xd[va])[:, 1]
        oof2 = np.stack([1 - oof, oof], 1)
        t, f1o, _ = calibrate_threshold(oof2, dev_labels)
        lr_full = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000).fit(Xd, dev_labels)
        pt = lr_full.predict_proba(Xt)[:, 1]
        results['stacking'] = {'t': t, 'f1': f1o, 'test': np.stack([1 - pt, pt], 1)}
        print(f"  {'stacking':12s}: Dev (out-of-fold) Macro-F1 {f1o:.4f} @tau*={t:.3f} | coef {np.round(lr_full.coef_[0], 2)}")

    best_name = max((k for k in results if k != 'stacking'), key=lambda k: results[k]['f1'])
    if 'stacking' in results and results['stacking']['f1'] > results[best_name]['f1'] + 0.01:
        best_name = 'stacking'
    best_single = max(member_f1, key=member_f1.get)
    if results[best_name]['f1'] < member_f1[best_single] - 0.005:
        print(f"  (no ensemble beats the best single model {best_single}; using it directly)")
        results['single'] = {'t': calibrate_threshold(RUN_PREDICTIONS_DEV[best_single], dev_labels)[0],
                             'f1': member_f1[best_single], 'test': RUN_PREDICTIONS_TEST[best_single]}
        best_name = 'single'
    R = results[best_name]
    print("=" * 60)
    print(f"Selected ensemble strategy: {best_name} | Dev Macro-F1 {R['f1']:.4f} @ tau*={R['t']:.3f}")
    print("=" * 60)
    RUN_METRICS_SUMMARY['Run 5 (Master Multi-Model Ensemble)'] = {'f1': R['f1'], 'threshold': R['t']}
    package_submission(R['test'], test_ids, R['t'], ENSEMBLE_DIR, "submission_master_ensemble.zip")

In [ ]:
# Final results summary & submission check
for d in [OUT_DIR] + list(PREV_OUT_DIRS):
    for p in sorted(glob.glob(os.path.join(d, "*_meta.json"))):
        m = json.load(open(p))
        if m['label'] not in RUN_METRICS_SUMMARY:
            RUN_METRICS_SUMMARY[m['label']] = {'f1': m['f1'], 'threshold': m['threshold']}
print("=" * 75)
print(f"{'Experiment':<38} | {'Threshold':<10} | {'Dev Macro-F1':<12}")
print("=" * 75)
for name, data in RUN_METRICS_SUMMARY.items():
    print(f"{name:<38} | {data['threshold']:<10.3f} | {data['f1']:<12.4f}")
print("=" * 75)

zip_files = ["submission_run1_arch4_lorma_BEST.zip", "submission_run2_arch3_adalora.zip",
             "submission_run3_arch2_layerwise.zip", "submission_run4_kd_arch4_to_arch1.zip",
             "submission_master_ensemble.zip"]
for zf_name in zip_files:
    full_p = os.path.join(WORK_DIR, zf_name)
    if os.path.exists(full_p):
        with zipfile.ZipFile(full_p, 'r') as z:
            recs = json.load(z.open("prediction.json"))
        c1 = sum(r['pred'] for r in recs)
        print(f"✓ {zf_name:40s}: {len(recs)} preds (Class 0: {len(recs)-c1}, Class 1: {c1}) | {os.path.getsize(full_p)/1024:.1f} KB")
    else:
        print(f"- {zf_name} not generated in this session.")